# 07 - Categorical Encoding

Models work with numbers, not text, so every categorical column in this dataset (`Gender`, `ContractType`, `PaymentMethod`, `InternetService`, `TechSupport`, `PaperlessBilling`, `Churn`) needs to be converted into a numeric form before it can go into a model. This notebook is about picking the right conversion method per column, not applying the same one everywhere.

Working off a lightly cleaned version of the data - fixing the Gender/ContractType spelling inconsistencies first, since encoding messy categories just bakes the mess into more columns.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# fixing the inconsistent spellings flagged back in Notebook 1 and 5,
# before any encoding happens
gender_map = {"M": "Male", "male": "Male", "F": "Female"}
df["Gender"] = df["Gender"].replace(gender_map)

contract_map = {"month-to-month": "Month-to-month", "Monthly": "Month-to-month"}
df["ContractType"] = df["ContractType"].replace(contract_map)

print(df["Gender"].unique())
print(df["ContractType"].unique())

['Male' 'Female']
['Two Year' 'One Year' 'Month-to-Month']


## 1. What is Categorical Encoding

Categorical encoding is the process of turning category labels ("Male", "Fiber optic", "Two year") into numbers a model can actually use in its math. Different encoding methods make different assumptions about the category, so picking the wrong one can either lose information or invent a false relationship that doesn't exist in the real data.

## 2. Nominal vs Ordinal vs Binary Variables

- **Nominal** - categories with no natural order. `PaymentMethod` ("Electronic check", "Mailed check", "Bank transfer", "Credit card") - none of these is "bigger" or "better" than another, they're just different.
- **Ordinal** - categories with a genuine order. `ContractType` ("Month-to-month" < "One year" < "Two year") - there's a real progression here based on contract length.
- **Binary** - only two categories. `Gender` (Male/Female), `PaperlessBilling` (Yes/No), `Churn` (Yes/No).

Why this distinction matters: if I encode a nominal column with plain numbers (0, 1, 2, 3), the model will wrongly assume category 3 is "more" than category 0 in some mathematical sense, when really they're just different labels with no ranking. That's the core reason different encoding methods exist.

## 3. Label Encoding

Label encoding assigns each category an arbitrary integer (0, 1, 2, ...). It's simple, but it implies an order that may not exist. Using it here on `Churn` since that's the target column, and for a binary target, 0/1 is exactly what's needed anyway - no false ordering problem with only two categories.

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Churn_encoded"] = le.fit_transform(df["Churn"])

print(dict(zip(le.classes_, le.transform(le.classes_))))
df[["Churn", "Churn_encoded"]].head()

{'No': np.int64(0), 'Yes': np.int64(1)}


,Churn,Churn_encoded
0,Yes,1
1,No,0
2,No,0
3,Yes,1
4,Yes,1


## 4. Ordinal Encoding

Ordinal encoding also maps categories to integers, but the order is chosen deliberately to reflect a real ranking, rather than being alphabetical or arbitrary. Using this for `ContractType`, since "Month-to-month" < "One year" < "Two year" is a genuine progression, and a model might actually benefit from knowing that ordering (e.g. longer contracts tend to mean lower churn).

## 5. One-Hot Encoding and Dummy Variables

One-hot encoding creates a separate 0/1 column for each category, so there's no implied order at all - the safest choice for nominal data. `PaymentMethod` and `InternetService` are both nominal (no real ranking between "DSL" and "Fiber optic"), so this is the right fit for them.

"Dummy variables" is the same idea, but drops one category (`drop_first=True`) to avoid redundancy - if a row isn't Electronic check, Mailed check, or Bank transfer, it must be Credit card, so that fourth column doesn't add new information and can cause multicollinearity in linear models.

In [6]:
payment_encoded = pd.get_dummies(df["PaymentMethod"], prefix="Payment", drop_first=True)
internet_encoded = pd.get_dummies(df["InternetService"], prefix="Internet", drop_first=True)

payment_encoded.head()

,Payment_Credit Card,Payment_Electronic Check,Payment_Mailed Check
0,False,False,True
1,False,False,False
2,True,False,False
3,False,True,False
4,False,False,True


## 6. Frequency Encoding

Frequency encoding replaces each category with how often it appears in the data. It's a lighter-weight alternative to one-hot encoding when a column has a lot of categories, since it doesn't blow up the number of columns. Not strictly needed for any column in this dataset (none of them have that many categories), but showing it on `PaymentMethod` for comparison, since it's a case where either approach could reasonably apply.

In [7]:
freq_map = df["PaymentMethod"].value_counts(normalize=True)
df["PaymentMethod_freq"] = df["PaymentMethod"].map(freq_map)
df[["PaymentMethod", "PaymentMethod_freq"]].drop_duplicates()

,PaymentMethod,PaymentMethod_freq
0,Mailed Check,0.243281
1,Bank Transfer,0.233399
2,Credit Card,0.266798
3,Electronic Check,0.256522


## 7. Target Encoding

Target encoding replaces each category with the average target value (e.g. average churn rate) for that category. It can be powerful, but it's also the riskiest encoding method here, because it uses the target column to build the feature - if this is computed on the full dataset before a train/test split, it's a direct example of target leakage (which comes back properly in Notebook 13). Showing the mechanics on `ContractType` here, but noting clearly this must only ever be fit on training data in the real workflow.

In [8]:
churn_numeric = df["Churn"].map({"Yes": 1, "No": 0})
target_encoded_map = df.groupby("ContractType")["Churn"].apply(lambda x: (x == "Yes").mean())
print(target_encoded_map)

# NOTE: doing this on the full df here is just to show the mechanic.
# In the real pipeline this must be computed using ONLY the training
# split, then applied to validation/test - otherwise it leaks target
# information from rows the model shouldn't have seen yet.

ContractType
Month-to-Month    0.273579
One Year          0.251978
Two Year          0.260234
Name: Churn, dtype: float64


## 8. Handling Unknown Categories and High-Cardinality Categories

Unknown categories are values that show up at prediction time but weren't seen during training - e.g. a new payment method the business adds later. If an encoder isn't told how to handle this, it can crash. `OneHotEncoder` from scikit-learn (as opposed to `pd.get_dummies`) has a `handle_unknown="ignore"` option specifically for this, which is safer for production use than `pd.get_dummies`.

High-cardinality categories (a column with dozens or hundreds of unique values, like a raw zip code or product ID) blow up one-hot encoding into hundreds of columns. None of my columns are high-cardinality, but `CustomerID` would be an extreme example if I tried to encode it - which is exactly why it's an identifier to be dropped in feature selection, not encoded at all.

In [9]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(df[["PaymentMethod"]])

# simulate a category the encoder has never seen
new_data = pd.DataFrame({"PaymentMethod": ["Crypto wallet"]})
result = ohe.transform(new_data)
print(result)  # all zeros - unknown category handled without crashing, instead of raising an error

[[0. 0. 0. 0.]]
